In [2]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.contrib.concurrent import thread_map, process_map
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import sys
from pathlib import Path

# Get the absolute path of the current file
current_file_path = Path(os.getcwd())

# Get the parent directory of the current file
current_directory = current_file_path.parent

# Get the parent directory of the current directory (which is the "grandparent" or the parent of the project)
parent_directory = current_directory.parent

# Append the parent directory to sys.path
sys.path.append("../") # Or os.path.dirname(script_dir) if helper_code is up one level
from helper_code import load_label, load_source, find_records, find_records_abs, load_signals

In [4]:
signal, text = load_signals("/juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part2/2954482")

In [10]:
signal[5]

array([ 0.007,  0.006, -0.001, -0.007,  0.004,  0.003, -0.01 , -0.019,
        0.004, -0.007, -0.018,  0.005])

In [2]:
# Define the directory containing training data
training_data_dir = "../training_data"

# Find all records in the directory
records = find_records_abs(training_data_dir)
print(f"Found {len(records)} records.")

Found 366185 records.


In [6]:
load_label(records[-1])

1

In [25]:
records[0]

'/juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part0/1000010'

In [15]:
def extract_labels(records):
    # max_workers is passed directly to thread_map
    labels = thread_map(load_label, records,
                        total=len(records),
                        desc="Loading labels (contrib)",
                        max_workers=20)
    return list(labels) # thread_map returns a generator

def extract_sources(records):
    sources = thread_map(load_source, records,
                         total=len(records),
                         desc="Loading sources (contrib)",
                         max_workers=20)
    return list(sources)

    

labels = extract_labels(records)
sources = extract_sources(records)

# Combine labels and sources for stratification
stratify_data = [(source, label) for source, label in zip(sources, labels)]

Loading sources: 100%|██████████| 366185/366185 [01:07<00:00, 5389.46it/s]


In [18]:
stratify_data[-1]

('SaMi-Trop', 1)

In [19]:
# Perform stratified split into train, validation, and test sets
train_records, temp_records, train_stratify, temp_stratify = train_test_split(
    records, stratify_data, test_size=0.2, random_state=42
)
val_records, test_records, _, _ = train_test_split(
    temp_records, temp_stratify, test_size=0.5, random_state=42
)

print(f"Train records: {len(train_records)}")
print(f"Validation records: {len(val_records)}")
print(f"Test records: {len(test_records)}")

Train records: 292948
Validation records: 36618
Test records: 36619


In [28]:
# calculate number of pos and negative labels
def count_labels(labels):
    pos_count = sum(1 for label in labels if label == 1)
    neg_count = sum(1 for label in labels if label == 0)
    return pos_count, neg_count
print(count_labels(labels))
print("posweight: ", count_labels(labels)[1] / count_labels(labels)[0])


(8190, 357995)
posweight:  43.711233211233214


In [20]:
# Create a DataFrame for the split information
split_data = []
for record in train_records:
    split_data.append({"exam_id": os.path.basename(record), "split": "train"})
for record in val_records:
    split_data.append({"exam_id": os.path.basename(record), "split": "val"})
for record in test_records:
    split_data.append({"exam_id": os.path.basename(record), "split": "test"})

split_df = pd.DataFrame(split_data)

# Save the DataFrame to a CSV file
# output_csv_path = "split_info.csv"
# split_df.to_csv(output_csv_path, index=False)
# print(f"Split information saved to {output_csv_path}")

In [23]:
split_df.to_csv("train_val_test_sets.csv", index=False)

# Summary of Stratified Split
The dataset has been split into training, validation, and test sets using an 80-10-10 ratio. Stratification was performed based on both the source of the data and the Chagas label.

Additionally, a CSV file (`split_info.csv`) has been generated containing the `exam_id` and its corresponding split (`train`, `val`, or `test`).